In [1]:
from embedder import Embedder

embed = Embedder("models/Xenova/all-MiniLM-L6-v2")

In [2]:
## Q1 - first value if a vector
question = "How does approximate nearest neighbor search work?"
embbeding = embed.encode(question)

In [3]:
embbeding[0]

np.float64(-0.02058203437252893)

In [4]:
## review github data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [18]:
filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
document_to_encode = [i for i in documents if filename in i["filename"]][0]

In [19]:
document_embed = embed.encode(document_to_encode["content"])

In [20]:
## q2 - sim cons between embbeding and document_embed
embbeding.dot(document_embed)

np.float64(0.36107027225589694)

In [21]:
## for the Q# wewil use chunks
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
texts_chunked = [doc["content"] for doc in chunks]

In [27]:
## encoding using batch
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts_chunked), batch_size)):
    batch = texts_chunked[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [30]:
scores = X.dot(embbeding)

In [32]:
## Q3 file name of chunk higher score
idx = np.argmax(scores)

chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [33]:
from minsearch import VectorSearch

In [34]:
## load the numpy vectors and the documents
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [36]:
## Q4 - First result file name
vector2 =  embed.encode("What metric do we use to evaluate a search engine?")
results = vindex.search(vector2, num_results=1)

In [39]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [40]:
## search using text search
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.searching_tool import fit_docs

In [41]:
index = fit_docs(
    chunks,
    text_fields=["content"],
    keyword_fields=["filename"]
)

In [45]:
## Q5 - File in vector resuts but not in text results
query_q5 = "How do I store vectors in PostgreSQL?"
results_text = index.search(query_q5)[:5]
vector_q5 = embed.encode(query_q5)
results_embeddings = vindex.search(vector_q5, num_results=5)

In [48]:
filesnames_text = [i["filename"] for i in results_text]
filesnames_emb = [i["filename"] for i in results_embeddings]
[i for i in filesnames_emb if i not in filesnames_text]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [51]:
## hybrid search
# reciprocal rank fusion (RRF) function

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [52]:
query_q6 = "How do I give the model access to tools?"
text_results = index.search(query_q6)[:5]
vector_q6 = embed.encode(query_q6)
vector_results = vindex.search(vector_q6, num_results=5)
results = rrf([vector_results, text_results])

In [56]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'